In [1]:
import os, sys, subprocess, numpy as np, pandas as pd
from pathlib import Path
from typing import Tuple, Dict, List

# -------------------------
# Wheels (offline installs)
# -------------------------
RDKit_WHL_DIR = "/kaggle/input/rdkit-2025-3-3-cp311"
MORDRED_DIR   = "/kaggle/input/mordred-1-2-0-py3-none-any"
RDKIT_WHL     = f"{RDKit_WHL_DIR}/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl"
MORDRED_WHL   = f"{MORDRED_DIR}/mordred-1.2.0-py3-none-any.whl"
NX_WHL        = f"{MORDRED_DIR}/networkx-2.8.8-py3-none-any.whl"

def _pip(path, *extra):
    if os.path.exists(path):
        print(f"[Install] {os.path.basename(path)} {' '.join(extra)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *extra, path])

_pip(RDKIT_WHL)
_pip(NX_WHL)
_pip(MORDRED_WHL, "--no-deps")

# -------------------------
# Imports
# -------------------------
from rdkit import Chem
from mordred import Calculator, descriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.isotonic import IsotonicRegression

# Optional learners (default OFF to keep runtime low)
USE_EXTRA_TREES = False  # True to add LGB/XGB (adds ~30–60+ mins)
if USE_EXTRA_TREES:
    import lightgbm as lgb
    import xgboost as xgb

# -------------------------
# Config
# -------------------------
N_SPLITS = 5
RANDOM_SEEDS = [42, 2025]      # 2 seeds (good diversity, much faster)
BLEND_GRID_STEP = 0.05         # coarser grid = faster
DESC_DIR = "/kaggle/input/modred-dataset"
ECFP_BITS = 1024
USE_ECFP_R3 = False            # set True to add r=3 (slower)

# -------------------------
# Data discovery
# -------------------------
def _find_data_dir() -> str:
    for d in Path("/kaggle/input").glob("*"):
        if d.is_dir():
            name = d.name.lower()
            if any(k in name for k in ["polymer","neurips","open","opp"]) and (d/"train.csv").exists() and (d/"test.csv").exists():
                return str(d)
    return "/kaggle/input/neurips-open-polymer-prediction-2025"

DATA_DIR = _find_data_dir()
print("[DATA_DIR]", DATA_DIR)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

def _find_smiles(df: pd.DataFrame) -> str:
    for c in df.columns:
        if str(c).lower().strip() in ("smiles","smile","smiles_str","polymer_smiles"):
            return c
    raise KeyError("SMILES column not found in train.csv")

SMI_COL = _find_smiles(train)
TARGETS = ["Tg","FFV","Tc","Density","Rg"]

# Precomputed Mordred train descriptor tables (per-target curated)
tg      = pd.read_csv(f"{DESC_DIR}/desc_tg.csv",      low_memory=False)
tc      = pd.read_csv(f"{DESC_DIR}/desc_tc.csv",      low_memory=False)
rg      = pd.read_csv(f"{DESC_DIR}/desc_rg.csv",      low_memory=False)
ffv     = pd.read_csv(f"{DESC_DIR}/desc_ffv.csv",     low_memory=False)
density = pd.read_csv(f"{DESC_DIR}/desc_de.csv",      low_memory=False)

def _prep_train_df(df: pd.DataFrame, target: str) -> Tuple[pd.DataFrame, pd.Series]:
    smiles = df["SMILES"].astype(str) if "SMILES" in df.columns else pd.Series([None]*len(df))
    const = [c for c in df.columns if df[c].nunique(dropna=True) == 1]
    if const:
        df = df.drop(columns=const)
    y = pd.to_numeric(df[target], errors="coerce")
    X = df.drop(columns=[target], errors="ignore").select_dtypes(exclude=["object","category"])
    X[target] = y
    X = X.replace([np.inf, -np.inf], np.nan)
    return X, smiles

tg, tg_smiles             = _prep_train_df(tg, "Tg")
ffv, ffv_smiles           = _prep_train_df(ffv, "FFV")
tc, tc_smiles             = _prep_train_df(tc, "Tc")
density, density_smiles   = _prep_train_df(density, "Density")
rg, rg_smiles             = _prep_train_df(rg, "Rg")

print("tg shape:", tg.shape)
print("ffv shape:", ffv.shape)
print("tc shape:", tc.shape)
print("density shape:", density.shape)
print("rg shape:", rg.shape)

# -------------------------
# Mordred test descriptors
# -------------------------
print("[Mordred] Computing test descriptors (2D; ignore_3D=True)...")
mols_test = [Chem.MolFromSmiles(s) for s in test[SMI_COL].astype(str)]
desc_test_raw = Calculator(descriptors, ignore_3D=True).pandas(mols_test)
desc_test_raw = desc_test_raw.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
print("desc_test shape:", desc_test_raw.shape)

# -------------------------
# ECFP cache (compute ONCE for all unique SMILES)
# -------------------------
def _has_gpu() -> bool:
    if os.environ.get("CUDA_VISIBLE_DEVICES", "") in ("", "-1"):
        return False
    return True

_GEN_CACHE: Dict[Tuple[int,int], object] = {}
def _get_morgan_gen(radius: int, nBits: int):
    key = (radius, nBits)
    if key not in _GEN_CACHE:
        _GEN_CACHE[key] = GetMorganGenerator(radius=radius, countSimulation=True, fpSize=nBits)
    return _GEN_CACHE[key]

def _ecfp_counts_for_smiles(unique_smiles: pd.Index, radius: int, nBits: int) -> pd.DataFrame:
    gen = _get_morgan_gen(radius, nBits)
    mats = np.zeros((len(unique_smiles), nBits), dtype=np.float32)
    for i, s in enumerate(unique_smiles):
        m = Chem.MolFromSmiles(s)
        if m is None: 
            continue
        fp = gen.GetCountFingerprint(m) if hasattr(gen, "GetCountFingerprint") else gen.GetSparseCountFingerprint(m)
        for k, v in fp.GetNonzeroElements().items():
            mats[i, int(k) % nBits] = float(v)
    cols = [f"ECFP{radius}_C_{i}" for i in range(nBits)]
    df = pd.DataFrame(mats, index=unique_smiles, columns=cols)
    return df.astype(np.float16)  # half precision: faster & smaller

CACHE_DIR = Path("/kaggle/working")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

all_smiles = pd.Index(
    pd.concat([train[SMI_COL].astype(str), test[SMI_COL].astype(str)], ignore_index=True).unique()
)

def _load_or_build_ecfp(radius: int, nBits: int) -> pd.DataFrame:
    p = CACHE_DIR / f"ecfp_r{radius}_{nBits}.parquet"
    if p.exists():
        return pd.read_parquet(p)
    df = _ecfp_counts_for_smiles(all_smiles, radius, nBits)
    df.to_parquet(p, index=True)
    return df

print("[ECFP] Building/loading cached ECFP…")
ecfp2_all = _load_or_build_ecfp(2, ECFP_BITS)
ecfp3_all = _load_or_build_ecfp(3, ECFP_BITS) if USE_ECFP_R3 else None

def _ecfp_slice(smiles_series: pd.Series, r3: bool=False) -> pd.DataFrame:
    s = smiles_series.astype(str)
    part2 = ecfp2_all.reindex(s)
    if r3 and (ecfp3_all is not None):
        part3 = ecfp3_all.reindex(s)
        return pd.concat([part2, part3], axis=1)
    return part2

# -------------------------
# Alignment helper
# -------------------------
def _align(train_df: pd.DataFrame, test_desc: pd.DataFrame, target: str):
    feat_cols = sorted((set(train_df.columns) - {target}) & set(test_desc.columns))
    X  = train_df[feat_cols].copy()
    y  = train_df[target].copy()
    Xt = test_desc[feat_cols].copy()
    return X, y, Xt, feat_cols

# -------------------------
# CatBoost presets (leaner)
# -------------------------
def _cb_params(target: str, random_seed: int):
    big = target in ("Tg", "FFV")
    return dict(
        loss_function="MAE",
        depth=8,
        learning_rate=0.04 if big else 0.05,
        l2_leaf_reg=6.0,
        random_strength=0.5,
        bagging_temperature=1.0,
        iterations=3000 if big else 2000,  # trimmed
        od_type="Iter",
        od_wait=300 if big else 200,
        random_seed=random_seed,
        verbose=False,
        task_type="GPU" if _has_gpu() else "CPU",
        thread_count=-1,
        allow_writing_files=False,
    )

# -------------------------
# OOF + Test: CatBoost on Mordred+ECFP (using cached ECFP slices)
# -------------------------
def mordred_cb_oof_and_test(train_df: pd.DataFrame,
                            smiles_series: pd.Series,
                            target: str):
    keep = train_df[target].notna()
    tr_df = train_df.loc[keep].reset_index(drop=True)
    tr_smiles = smiles_series.loc[keep].reset_index(drop=True)

    X_m, y, Xt_m, _ = _align(tr_df, desc_test_raw, target)
    med = X_m.median(axis=0)
    X_m  = X_m.fillna(med).reset_index(drop=True)
    Xt_m = Xt_m.fillna(med).reset_index(drop=True)

    # one-shot ECFP slice
    ecfp_tr = _ecfp_slice(tr_smiles, r3=USE_ECFP_R3)
    ecfp_te = _ecfp_slice(test[SMI_COL], r3=USE_ECFP_R3).reset_index(drop=True)

    X  = pd.concat([X_m, ecfp_tr.reset_index(drop=True)], axis=1)
    Xt = pd.concat([Xt_m.reset_index(drop=True), ecfp_te], axis=1)
    Xt = Xt[X.columns]  # align

    # drop all-zero columns (common after hashing)
    nz = (X != 0).any(axis=0).values
    X = X.loc[:, nz]
    Xt = Xt.loc[:, nz]

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=123)
    oof = np.zeros(len(X), dtype=float)

    for (tr, va) in kf.split(X):
        model = CatBoostRegressor(**_cb_params(target, RANDOM_SEEDS[0]))
        model.fit(X.iloc[tr], y.iloc[tr],
                  eval_set=(X.iloc[va], y.iloc[va]),
                  use_best_model=True, verbose=False)
        oof[va] = model.predict(X.iloc[va])

    oof_mae = mean_absolute_error(y, oof)
    print(f"  [{target}] CatBoost+ECFP OOF MAE = {oof_mae:.6f}  (features: {X.shape[1]})")

    preds = []
    for seed in RANDOM_SEEDS:
        m = CatBoostRegressor(**_cb_params(target, seed))
        m.fit(X, y, eval_set=(X, y), use_best_model=True, verbose=False)
        preds.append(m.predict(Xt))
    pred_test = np.mean(np.vstack(preds), axis=0)
    return oof, pred_test, tr_smiles, X, y, Xt

# -------------------------
# Optional: LGB/XGB on same features
# -------------------------
def _lgb_oof_test(X, y, Xt):
    # Robust imports across LightGBM versions
    try:
        from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping, log_evaluation as lgb_log_evaluation
    except Exception:
        import lightgbm as lgb
        LGBMRegressor = lgb.LGBMRegressor
        lgb_early_stopping = getattr(lgb, "early_stopping", None)
        lgb_log_evaluation = getattr(lgb, "log_evaluation", None)

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=123)
    oof = np.zeros(len(X), dtype=float); preds = []

    params = dict(
        objective="mae",
        n_estimators=8000,
        learning_rate=0.04,
        num_leaves=256,
        subsample=0.7,          # bagging_fraction
        colsample_bytree=0.7,   # feature_fraction
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_samples=20,
        n_jobs=-1,
        # use 'verbosity' in params, not 'verbose' in fit
        verbosity=-1,
    )

    for tr, va in kf.split(X):
        m = LGBMRegressor(**params)
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        Xva, yva = X.iloc[va], y.iloc[va]
        try:
            # Newer wrappers often accept both args below
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], early_stopping_rounds=300)
        except TypeError:
            # Fallback for versions without those kwargs
            callbacks = []
            if lgb_early_stopping is not None:
                callbacks.append(lgb_early_stopping(stopping_rounds=300, verbose=False))
            if lgb_log_evaluation is not None:
                callbacks.append(lgb_log_evaluation(period=0))  # silence logs
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=callbacks)

        best_it = getattr(m, "best_iteration_", None)
        oof[va] = m.predict(Xva, num_iteration=best_it)
        preds.append(m.predict(Xt, num_iteration=best_it))

    return oof, np.mean(np.vstack(preds), axis=0)


def _xgb_oof_test(X, y, Xt, target_name=""):
    import xgboost as xgb
    use_gpu = _has_gpu()
    kf = _kf_for_target(target_name) if target_name else KFold(n_splits=N_SPLITS, shuffle=True, random_state=123)
    oof = np.zeros(len(X), dtype=float); preds = []

    params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        max_depth=8,                 # ↓ from 10
        eta=0.05,                    # a bit faster convergence
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=1.0,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="gpu_hist" if use_gpu else "hist",
        predictor="gpu_predictor" if use_gpu else "auto",
        n_estimators=3000,           # ↓ from 8000
        early_stopping_rounds=200    # set in ctor (no deprecation warning)
    )

    for tr, va in kf.split(X):
        m = xgb.XGBRegressor(**params)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], verbose=False)
        oof[va] = m.predict(X.iloc[va])
        preds.append(m.predict(Xt))
    return oof, np.mean(np.vstack(preds), axis=0)

# -------------------------
# External sources (ChemBERTa / FastSMILES)
# -------------------------
def _discover_inputs():
    chem_root = None
    fast_root = None
    chem_sub = None
    fast_sub = None
    base = "/kaggle/input"
    for slug in os.listdir(base):
        slug_path = os.path.join(base, slug)
        if not os.path.isdir(slug_path):
            continue
        for r, d, f in os.walk(slug_path):
            fl = [x.lower() for x in f]
            if "chemberta_submission.csv" in fl:
                chem_sub = os.path.join(r, "chemberta_submission.csv")
            if "fastsmiles_submission.csv" in fl:
                fast_sub = os.path.join(r, "fastsmiles_submission.csv")
            if any(x.startswith("chemberta_a_") and x.endswith("_oof_preds.csv") for x in fl):
                chem_root = slug_path
            if any(x in ("ctb_a_oof_preds.csv","lgb_a_oof_preds.csv","xgb_a_oof_preds.csv") for x in fl):
                fast_root = slug_path
    return chem_root, fast_root, chem_sub, fast_sub

CHEMBERTA_ROOT, FASTSMILES_ROOT, CHEMBERTA_SUB, FASTSMILES_SUB = _discover_inputs()
print("[AUTO] CHEMBERTA_ROOT:", CHEMBERTA_ROOT)
print("[AUTO] FASTSMILES_ROOT:", FASTSMILES_ROOT)
print("[AUTO] CHEMBERTA_SUB:", CHEMBERTA_SUB)
print("[AUTO] FASTSMILES_SUB:", FASTSMILES_SUB)

def _align_oof_by_id(oof_csv_path: str, target_name: str) -> np.ndarray:
    df = pd.read_csv(oof_csv_path)
    cols = {c.lower(): c for c in df.columns}

    # Prefer exact id alignment
    if "id" in cols:
        val_col = cols.get(target_name.lower()) or cols.get("oof_preds")
        if val_col is None:
            # fallback: first numeric col (not id)
            for c in df.columns:
                if c != cols["id"] and pd.api.types.is_numeric_dtype(df[c]):
                    val_col = c; break
        if val_col is not None:
            ser = pd.Series(pd.to_numeric(df[val_col], errors="coerce").values,
                            index=df[cols["id"]].values)
            return ser.reindex(train["id"].values).values

    # Fallback: align via SMILES if present
    if "smiles" in cols:
        key = cols["smiles"]
        val_col = cols.get(target_name.lower())
        if val_col is None:
            for c in df.columns:
                if c != key and pd.api.types.is_numeric_dtype(df[c]):
                    val_col = c; break
        if val_col is not None:
            ser = pd.Series(pd.to_numeric(df[val_col], errors="coerce").values,
                            index=df[key].astype(str).values)
            return ser.reindex(train[SMI_COL].astype(str).values).values

    # Last resort: try sibling ids.csv / train_ids.csv / train.csv in the same folder
    here = Path(oof_csv_path).parent
    for extra in ("ids.csv", "train_ids.csv", "train.csv"):
        p = here / extra
        if p.exists():
            ids = pd.read_csv(p)
            idcol = next(c for c in ids.columns if c.lower()=="id")
            numcol = next(c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]))
            ser = pd.Series(pd.to_numeric(df[numcol], errors="coerce").values,
                            index=ids[idcol].values)
            return ser.reindex(train["id"].values).values

    # Ensure correct length even if nothing matched
    return np.full(len(train), np.nan, dtype=float)


def _align_submission_to_test(df_sub: pd.DataFrame, target_name: str) -> np.ndarray:
    if "id" in df_sub.columns and target_name in df_sub.columns:
        s = pd.Series(df_sub[target_name].values, index=df_sub["id"].values)
        return s.reindex(test["id"].values).values
    return df_sub[target_name].values

def _maybe_load_oof_chemberta_auto() -> Dict[str, np.ndarray]:
    out = {}
    if not CHEMBERTA_ROOT or not os.path.exists(CHEMBERTA_ROOT):
        return out
    for t in TARGETS:
        base = None
        for d in os.listdir(CHEMBERTA_ROOT):
            p = os.path.join(CHEMBERTA_ROOT, d)
            if os.path.isdir(p) and t.lower() in d.lower():
                base = p; break
        if base is None: 
            continue
        cand = None
        for f in os.listdir(base):
            if f.lower().endswith(".csv") and "oof" in f.lower():
                cand = os.path.join(base, f); break
        if cand:
            out[t] = _align_oof_by_id(cand, t)
    return out

def _maybe_load_oof_fastsmiles_auto(weights=(0.4,0.3,0.3)) -> Dict[str, np.ndarray]:
    out = {}
    if not FASTSMILES_ROOT or not os.path.exists(FASTSMILES_ROOT):
        return out
    for t in TARGETS:
        tdir = os.path.join(FASTSMILES_ROOT, t)
        if not os.path.isdir(tdir):
            continue
        parts = {}
        for sub in os.listdir(tdir):
            base = os.path.join(tdir, sub)
            if not os.path.isdir(base): 
                continue
            csv = None
            for f in os.listdir(base):
                if f.lower().endswith(".csv") and "oof" in f.lower():
                    csv = os.path.join(base, f); break
            if not csv: continue
            vec = _align_oof_by_id(csv, t)
            if sub.upper().startswith("CTB"): parts["CTB"] = vec
            if sub.upper().startswith("LGB"): parts["LGB"] = vec
            if sub.upper().startswith("XGB"): parts["XGB"] = vec
        if all(k in parts for k in ("CTB","LGB","XGB")):
            w = weights
            out[t] = w[0]*parts["CTB"] + w[1]*parts["LGB"] + w[2]*parts["XGB"]
    return out

def _oof_coverage(name: str, vec: np.ndarray):
    k = int(np.isfinite(vec).sum())
    print(f"[{name}] OOF aligned: {k}/{len(train)} non-NaN")


OOF_CHEM = _maybe_load_oof_chemberta_auto()
OOF_FAST = _maybe_load_oof_fastsmiles_auto(weights=(0.4,0.3,0.3))

for t in TARGETS:
    if t in OOF_CHEM:
        _oof_coverage(f"chem:{t}", OOF_CHEM[t])
    if t in OOF_FAST:
        _oof_coverage(f"fast:{t}", OOF_FAST[t])

TEST_CHEM = pd.read_csv(CHEMBERTA_SUB) if CHEMBERTA_SUB and os.path.exists(CHEMBERTA_SUB) else None
TEST_FAST = pd.read_csv(FASTSMILES_SUB) if FASTSMILES_SUB and os.path.exists(FASTSMILES_SUB) else None

have_chem = (len(OOF_CHEM)==5) and (TEST_CHEM is not None)
have_fast = (len(OOF_FAST)==5) and (TEST_FAST is not None)
print(f"[Sources] ChemBERTa {'✅' if have_chem else '—'}  FastSMILES {'✅' if have_fast else '—'}")

# -------------------------
# Train Mordred+ECFP CatBoost (+optional LGB/XGB)
# -------------------------
OOF_CB, TEST_CB, SMILES_NONNULL = {}, {}, {}
OOF_LGB, TEST_LGB = {}, {}
OOF_XGB, TEST_XGB = {}, {}

for (df, smi, t) in [(tg, tg_smiles, "Tg"),
                     (ffv, ffv_smiles, "FFV"),
                     (tc, tc_smiles, "Tc"),
                     (density, density_smiles, "Density"),
                     (rg, rg_smiles, "Rg")]:

    oof_cb, pred_cb, sm, X_mat, y_vec, Xt_mat = mordred_cb_oof_and_test(df, smi, t)
    OOF_CB[t], TEST_CB[t], SMILES_NONNULL[t] = oof_cb, pred_cb, sm

    if USE_EXTRA_TREES:
        oof_lgb, pred_lgb = _lgb_oof_test(X_mat, y_vec, Xt_mat)
        OOF_LGB[t], TEST_LGB[t] = oof_lgb, pred_lgb

        oof_xgb, pred_xgb = _xgb_oof_test(X_mat, y_vec, Xt_mat)
        OOF_XGB[t], TEST_XGB[t] = oof_xgb, pred_xgb

# -------------------------
# Broadcast OOFs to full train length via SMILES
# -------------------------
def _broadcast_oof_by_smiles(oof_vec: np.ndarray, smi_series: pd.Series, t: str) -> np.ndarray:
    full = np.full(len(train), np.nan, dtype=float)
    smi = smi_series.astype(str).values
    for val, s in zip(oof_vec, smi):
        mask = (train[SMI_COL].astype(str).values == s) & train[t].notna().values
        if mask.any():
            full[mask] = val
    return full

MORDRED_OOF_FULL = {t: _broadcast_oof_by_smiles(OOF_CB[t], SMILES_NONNULL[t], t) for t in TARGETS}
LGB_OOF_FULL = {t: _broadcast_oof_by_smiles(OOF_LGB[t], SMILES_NONNULL[t], t) for t in OOF_LGB} if USE_EXTRA_TREES else {}
XGB_OOF_FULL = {t: _broadcast_oof_by_smiles(OOF_XGB[t], SMILES_NONNULL[t], t) for t in OOF_XGB} if USE_EXTRA_TREES else {}

# -------------------------
# Calibration (Isotonic)
# -------------------------
def _calibrate(oof: np.ndarray, y: np.ndarray, pred_test: np.ndarray) -> np.ndarray:
    mask = (~np.isnan(oof)) & (~np.isnan(y))
    if mask.sum() < 100:
        return pred_test
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(oof[mask], y[mask])
    return ir.predict(pred_test)

def _maybe_calibrate_sources(t: str, srcs: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    y = pd.to_numeric(train[t].values, errors="coerce")
    out = {}
    for name, (oof, pred) in srcs.items():
        out[name] = (oof, _calibrate(oof, y, pred) if (oof is not None and pred is not None) else pred)
    return out

# -------------------------
# Blending (top-k by OOF, grid)
# -------------------------
def best_weights_for_target_safe(t: str,
                                 sources: Dict[str, Tuple[np.ndarray, np.ndarray]],
                                 step: float = 0.05):
    names = list(sources.keys())
    y_true = pd.to_numeric(train[t], errors="coerce").to_numpy()
    valid = ~np.isnan(y_true)

    keep, per_mae = [], {}
    for n in names:
        oof, _ = sources[n]
        if oof is not None and len(oof) == len(y_true):
            m = valid & (~np.isnan(oof))
            if m.sum() > 50:
                per_mae[n] = mean_absolute_error(y_true[m], oof[m])
                keep.append(n)
    if not keep:
        nn = [n for n in names if sources[n][1] is not None]
        w = {n: 1.0/len(nn) for n in nn} if nn else {}
        test_pred = np.mean([sources[n][1] for n in nn], axis=0) if nn else np.zeros(len(test))
        return w, test_pred

    keep = sorted(keep, key=lambda n: per_mae[n])[:3]
    oofs  = [sources[n][0] for n in keep]
    tests = [sources[n][1] for n in keep]

    mask = valid.copy()
    for o in oofs: mask &= ~np.isnan(o)
    yt = y_true[mask]
    oofs_m = [o[mask] for o in oofs]

    if len(keep) == 1:
        return {keep[0]: 1.0}, tests[0]

    ws = np.arange(0.0, 1.0 + 1e-9, step)
    best_mae, best_w = 1e9, None

    if len(keep) == 2:
        for w0 in ws:
            w1 = 1.0 - w0
            mae = mean_absolute_error(yt, w0*oofs_m[0] + w1*oofs_m[1])
            if mae < best_mae: best_mae, best_w = mae, [w0, w1]
        return dict(zip(keep, best_w)), best_w[0]*tests[0] + best_w[1]*tests[1]

    for w0 in ws:
        for w1 in ws:
            w2 = 1.0 - w0 - w1
            if w2 < 0: continue
            mae = mean_absolute_error(yt, w0*oofs_m[0] + w1*oofs_m[1] + w2*oofs_m[2])
            if mae < best_mae: best_mae, best_w = mae, [w0, w1, w2]
    return dict(zip(keep, best_w)), best_w[0]*tests[0] + best_w[1]*tests[1] + best_w[2]*tests[2]

# -------------------------
# Build sources, calibrate, blend
# -------------------------
FINAL = {"id": test["id"].values}
WEIGHTS = {}

PHYS_CLIP = {
    "FFV": (0.0, 1.0),
    "Density": (train["Density"].quantile(0.001), train["Density"].quantile(0.999)),
    "Tc": (train["Tc"].quantile(0.001), train["Tc"].quantile(0.999)),
    "Tg": (train["Tg"].quantile(0.001), train["Tg"].quantile(0.999)),
    "Rg": (train["Rg"].quantile(0.001), train["Rg"].quantile(0.999)),
}

for t in TARGETS:
    srcs: Dict[str, Tuple[np.ndarray, np.ndarray]] = {
        "cb":   (_broadcast_oof_by_smiles(OOF_CB[t], SMILES_NONNULL[t], t), TEST_CB[t]),
    }
    if USE_EXTRA_TREES:
        if t in OOF_LGB: srcs["lgb"] = (_broadcast_oof_by_smiles(OOF_LGB[t], SMILES_NONNULL[t], t), TEST_LGB.get(t))
        if t in OOF_XGB: srcs["xgb"] = (_broadcast_oof_by_smiles(OOF_XGB[t], SMILES_NONNULL[t], t), TEST_XGB.get(t))
    if have_chem:
        chem_oof = OOF_CHEM[t]
        chem_test = _align_submission_to_test(TEST_CHEM, t)
        srcs["chem"] = (chem_oof, chem_test)
    if have_fast:
        fast_oof = OOF_FAST[t]
        fast_test = _align_submission_to_test(TEST_FAST, t)
        srcs["fast"] = (fast_oof, fast_test)

    srcs = _maybe_calibrate_sources(t, srcs)
    w, pred = best_weights_for_target_safe(t, srcs, step=BLEND_GRID_STEP)

    lo, hi = PHYS_CLIP[t]
    pred = np.clip(pred, lo, hi)
    FINAL[t] = pred
    WEIGHTS[t] = w
    print(f"[Blend {t}] " + " + ".join(f"{w[k]:.2f}*{k}" for k in w))

# === CV vs LB sanity check ===
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

def _target_scale_stats(df_train: pd.DataFrame) -> dict:
    stats = {}
    for t in TARGETS:
        y = pd.to_numeric(df_train[t], errors="coerce")
        stats[t] = {
            "std": float(y.std(skipna=True)),
            "iqr": float(y.quantile(0.75) - y.quantile(0.25)),
            "rng": float(y.max(skipna=True) - y.min(skipna=True)),
        }
    return stats

SCALES = _target_scale_stats(train)

rows = []
for t in TARGETS:
    y = pd.to_numeric(train[t], errors="coerce").values
    valid = ~np.isnan(y)

    # Map source name -> OOF array (aligned to train)
    pool = {"cb": MORDRED_OOF_FULL[t]}
    if USE_EXTRA_TREES:
        if t in LGB_OOF_FULL: pool["lgb"] = LGB_OOF_FULL[t]
        if t in XGB_OOF_FULL: pool["xgb"] = XGB_OOF_FULL[t]
    if have_chem: pool["chem"] = OOF_CHEM[t]
    if have_fast: pool["fast"] = OOF_FAST[t]

    # Use the sources actually chosen by the blender
    used_names = list(WEIGHTS[t].keys())
    mats, ws = [], []
    for name in used_names:
        oof = pool.get(name, None)
        if oof is None or len(oof) != len(y):
            continue
        mats.append(oof)
        ws.append(WEIGHTS[t][name])

    if not mats:
        continue

    M = np.vstack(mats)                 # shape: (#sources_used, N)
    ws = np.asarray(ws, dtype=float)    # shape: (#sources_used,)
    mask = valid & ~np.isnan(M).any(axis=0)
    blended_oof = (ws[:, None] * M).sum(axis=0)

    cv_mae = mean_absolute_error(y[mask], blended_oof[mask])

    std = SCALES[t]["std"]
    iqr = SCALES[t]["iqr"]
    rng = SCALES[t]["rng"]

    rows.append({
        "target": t,
        "cv_mae": cv_mae,
        "cv_mae/std":  (cv_mae / std) if std > 0 else np.nan,
        "cv_mae/iqr":  (cv_mae / iqr) if iqr > 0 else np.nan,
        "cv_mae/range": (cv_mae / rng) if (rng and rng > 0) else np.nan,
        "used": " + ".join(f"{k}:{WEIGHTS[t][k]:.2f}" for k in used_names),
    })

cv_df = pd.DataFrame(rows)
print("\n[CV summary per target]")
print(cv_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# One-number proxy to compare with LB
proxy = cv_df["cv_mae/iqr"].mean()
LB_PUBLIC = 0.070  # your public LB score to compare against
print(f"\n[CV→LB proxy] mean(MAE/IQR) across targets = {proxy:.6f}")
print(f"[Compare] proxy {proxy:.6f} vs LB {LB_PUBLIC:.6f}  (Δ = {abs(proxy - LB_PUBLIC):.6f})")

sub = pd.DataFrame(FINAL, columns=["id","Tg","FFV","Tc","Density","Rg"])
out_path = "/kaggle/working/submission.csv"
sub.to_csv(out_path, index=False)
print("[Saved]", out_path)

[Install] rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl 
[Install] networkx-2.8.8-py3-none-any.whl 


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_

[Install] mordred-1.2.0-py3-none-any.whl --no-deps
[DATA_DIR] /kaggle/input/neurips-open-polymer-prediction-2025
tg shape: (9117, 594)
ffv shape: (7905, 597)
tc shape: (886, 566)
density shape: (1927, 473)
rg shape: (614, 545)
[Mordred] Computing test descriptors (2D; ignore_3D=True)...


100%|██████████| 3/3 [00:00<00:00,  5.45it/s]


desc_test shape: (3, 1613)
[ECFP] Building/loading cached ECFP…
[AUTO] CHEMBERTA_ROOT: /kaggle/input/neurips-opp2025-a-chemberta-t
[AUTO] FASTSMILES_ROOT: /kaggle/input/neurips-opp2025-a-fastsmiles-t
[AUTO] CHEMBERTA_SUB: /kaggle/input/neurips-opp2025-a-ensemble-0-068-lb/chemberta_submission.csv
[AUTO] FASTSMILES_SUB: /kaggle/input/neurips-opp2025-a-ensemble-0-068-lb/fastsmiles_submission.csv
[chem:Tg] OOF aligned: 0/7973 non-NaN
[fast:Tg] OOF aligned: 0/7973 non-NaN
[chem:FFV] OOF aligned: 0/7973 non-NaN
[fast:FFV] OOF aligned: 0/7973 non-NaN
[chem:Tc] OOF aligned: 0/7973 non-NaN
[fast:Tc] OOF aligned: 0/7973 non-NaN
[chem:Density] OOF aligned: 0/7973 non-NaN
[fast:Density] OOF aligned: 0/7973 non-NaN
[chem:Rg] OOF aligned: 0/7973 non-NaN
[fast:Rg] OOF aligned: 0/7973 non-NaN
[Sources] ChemBERTa ✅  FastSMILES ✅
  [Tg] CatBoost+ECFP OOF MAE = 25.093705  (features: 1617)
  [FFV] CatBoost+ECFP OOF MAE = 0.005667  (features: 1620)
  [Tc] CatBoost+ECFP OOF MAE = 0.030450  (features: 1589)
